## Versión final organizada

Este notebook conserva los experimentos estadísticos z-control que se retiraron del notebook individual para mantenerlo breve. Los resultados siguen escribiéndose en el mismo directorio del experimento.


In [ ]:
import importlib
import project_paths as _project_paths
_project_paths = importlib.reload(_project_paths)
from project_paths import (
    PROJECT_ROOT, ANALYSIS_DATA_DIR, OSSE_DATA_DIR,
    cse_experiment_paths, ensure_output_directories,
)
ensure_output_directories()
EXPERIMENT_NAME = 'z_control_nature_qiwenevents'
EXPERIMENT_ROOT, RESULTS_DIR, EXPERIMENT_FIGURES_DIR = cse_experiment_paths(EXPERIMENT_NAME)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)


# Qiwen CSE z-only control — Batch experiments


Este notebook ejecuta de forma autónoma:

- las 40 realizaciones para cada amplitud `D`;
- la rejilla completa T×D de la Figura 3(g);
- el resumen y la comparación con los valores publicados por Qiwen.

Las celdas guardan resultados parciales y pueden reanudarse.


## Imports


In [ ]:
import importlib
import pickle
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from class_da_sys import da_system
import class_cse_qiwen as cse_qiwen

cse_qiwen = importlib.reload(cse_qiwen)
from class_cse_qiwen import detect_regime_changes, run_single_cse as run_single_cse_core


## DA System Setup


In [ ]:
BASE_PATH = PROJECT_ROOT
PathFigures = EXPERIMENT_FIGURES_DIR
PathFigures.mkdir(parents=True, exist_ok=True)

# Both variants assimilate x, y, z. This variant restricts each CSE
# perturbation to the z component; x and y receive no direct increment.
CONTROL_INDICES = [2]
CONTROL_LABEL = 'z'
REGIME_REFERENCE_MODE = 'nature'
EVENT_SELECTION = 'qiwen'
OUTPUT_TAG = '_qiwenevents'
multiple_number = 4
T_0 = 75.1
position = int(np.ceil(T_0 * multiple_number))

das = da_system().load(ANALYSIS_DATA_DIR / 'x_analysis_init.pkl')

sv = das.getStateVector()
x_nature = sv.getTrajectory()

obs = das.getObsData()
y_obs = obs.getVal()

Ta = das.acyc_step
fcst_dt = das.fcst_dt

das.multiplicative_inflation = 1.04


## Load OSSE Histories


In [ ]:
with open(OSSE_DATA_DIR / 'osse_Xa_history.pkl', 'rb') as f:
    osse_Xa_history = pickle.load(f)

with open(OSSE_DATA_DIR / 'osse_summary.pkl', 'rb') as f:
    osse_summary = pickle.load(f)

print('osse_Xa_history shape:', osse_Xa_history.shape)
print("osse_summary['t0']:", osse_summary['t0'])
print("osse_summary['num_cycles']:", osse_summary['num_cycles'])


## Detect Regime Changes


In [ ]:
# Detect every regime transition in the NATURE trajectory as a sign change
# of the x component (a crossing of x=0 between two consecutive time steps).
# With EVENT_SELECTION='qiwen', the function accepts crossings in both
# directions, keeps those occurring after step 500 whose preceding residence
# time is longer than 385 steps, and retains at most the first 100 events.
# The complete list of crossings is also saved in RESULTS_DIR for reuse.
regime_change_info = detect_regime_changes(
    x_nature,
    RESULTS_DIR,
    position=position,
    output_filename='change_region_time_qiwen_qiwenevents.pkl',
    event_selection=EVENT_SELECTION,
)

# All detected x=0 crossings in the NATURE run.
change_region_time = regime_change_info['change_region_time']
# Forecast/control horizon used by the function's diagnostic sign check.
check_position = regime_change_info['check_position']
# Filtered crossing times used as the CSE event set.
t_start_series = regime_change_info['t_start_series']
# Time spent in the previous regime before each selected crossing.
t_start_residence = regime_change_info['t_start_residence']
# Mean interval between all detected crossings, before event filtering.
mean_crossing_interval = regime_change_info['mean_crossing_interval']


## Batch Parameters


In [ ]:
# Keep the shared CSE defaults consistent with the individual-event notebook.
CONTROL_INDICES = [2]
CONTROL_LABEL = 'z'
REGIME_REFERENCE_MODE = 'nature'
EVENT_SELECTION = 'qiwen'
OUTPUT_TAG = '_qiwenevents'
multiple_number = 4
T_0 = 75.1
position = int(np.ceil(T_0 * multiple_number))
experiment_step = 1000

print('position:', position)


## Reusable Single-Event Runner


In [ ]:
# Thin notebook wrapper around the single authoritative implementation.
def run_single_cse(event_index, D_run, seed, regime_reference='nature', candidate_pool='full'):
    return run_single_cse_core(
        event_index=event_index,
        D_run=D_run,
        seed=seed,
        das=das,
        x_nature=x_nature,
        t_start_series=t_start_series,
        osse_Xa_history=osse_Xa_history,
        osse_summary=osse_summary,
        experiment_step=experiment_step,
        position=position,
        Ta=Ta,
        fcst_dt=fcst_dt,
        path_figures=PathFigures,
        total_events=40,
        regime_reference=regime_reference,
        candidate_pool=candidate_pool,
        control_indices=CONTROL_INDICES,
        T_0=T_0,
        multiple_number=multiple_number,
    )


## Diagnostics Preserved

Cada resultado conserva las trayectorias, observaciones CSE, ramas de control, intervenciones, RMSE del análisis y registro de búsquedas pasadas seguras. Los diagnósticos visuales de desarrollo se retiraron del flujo principal, pero permanecen recuperables en la copia de seguridad funcional.


## Batch Run: 40 CSE Events


In [ ]:
# Run the z-control CSE validation over the first 40 selected events
# for every configured perturbation amplitude D. The pickle for each D is
# updated after every event, so interrupted runs can resume safely.
import pickle
import time

D_VALUES = [0.02, 0.03, 0.04, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
REGIME_REFERENCE = 'nature'
#CANDIDATE_POOL = 'unchanged_bkp'
CANDIDATE_POOL = 'full'
N_BATCH_EVENTS = 40

for D_run in D_VALUES:
    output_suffix = '' if (REGIME_REFERENCE == 'nature' and CANDIDATE_POOL == 'full') else f'_{REGIME_REFERENCE}_{CANDIDATE_POOL}'
    output_file = RESULTS_DIR / f'cse_qiwen_z_control_40_results_D{D_run}{output_suffix}{OUTPUT_TAG}.pkl'
    if output_file.exists():
        with open(output_file, 'rb') as f:
            results = pickle.load(f)
        print(f'Resuming {output_file}: {len(results)}/{N_BATCH_EVENTS} events already done')
    else:
        results = []

    for i in range(len(results), N_BATCH_EVENTS):
        t_event_start = time.time()
        results.append(run_single_cse(
            i,
            D_run,
            seed=1000 + i,
            regime_reference=REGIME_REFERENCE,
            candidate_pool=CANDIDATE_POOL,
        ))
        with open(output_file, 'wb') as f:
            pickle.dump(results, f)
        elapsed_min = (time.time() - t_event_start) / 60.0
        print(f'Saved partial result {i + 1}/{N_BATCH_EVENTS} for D={D_run} ({elapsed_min:.2f} min)')

    print('Saved:', output_file)


## Batch Run: Figure 3(g) T-D Grid


In [ ]:
# Run the z-control validation over the full Figure 3(g) grid: rows T/T0 and columns D.
# This is resumable: every (T, D) pair is stored in its own pickle and updated after each event.
import pickle
import time

FIG3G_T_MULTIPLES = {
    '[1.5T0]': 1.5,
    '[2T0]': 2.0,
    '[2.5T0]': 2.5,
    '[3T0]': 3.0,
    '[4T0]': 4.0,
}
FIG3G_D_VALUES = [0.02, 0.03, 0.04, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
FIG3G_N_BATCH_EVENTS = 40
FIG3G_REGIME_REFERENCE = 'nature'
FIG3G_CANDIDATE_POOL = 'full'


def _fig3g_t_file_label(T_label):
    return (
        T_label.strip('[]')
        .replace('.', 'p')
        .replace('T0', 'T0')
    )


def _fig3g_output_file(T_label, D_run):
    T_file_label = _fig3g_t_file_label(T_label)
    output_suffix = (
        ''
        if (FIG3G_REGIME_REFERENCE == 'nature' and FIG3G_CANDIDATE_POOL == 'full')
        else f'_{FIG3G_REGIME_REFERENCE}_{FIG3G_CANDIDATE_POOL}'
    )
    return RESULTS_DIR / f'cse_qiwen_z_control_40_results_{T_file_label}_D{D_run}{output_suffix}{OUTPUT_TAG}.pkl'


def run_single_cse_fig3g(event_index, D_run, seed, position_run, T_label, multiple_run):
    row = run_single_cse_core(
        event_index=event_index,
        D_run=D_run,
        seed=seed,
        das=das,
        x_nature=x_nature,
        t_start_series=t_start_series,
        osse_Xa_history=osse_Xa_history,
        osse_summary=osse_summary,
        experiment_step=experiment_step,
        position=position_run,
        Ta=Ta,
        fcst_dt=fcst_dt,
        path_figures=PathFigures,
        total_events=FIG3G_N_BATCH_EVENTS,
        regime_reference=FIG3G_REGIME_REFERENCE,
        candidate_pool=FIG3G_CANDIDATE_POOL,
        control_indices=CONTROL_INDICES,
    )
    row['T_label'] = T_label
    row['multiple_number'] = float(multiple_run)
    row['position'] = int(position_run)
    return row


for T_label, multiple_run in FIG3G_T_MULTIPLES.items():
    position_run = int(np.ceil(T_0 * multiple_run))
    print(f'=== T={T_label}, multiple={multiple_run}, position={position_run} ===')
    for D_run in FIG3G_D_VALUES:
        output_file = _fig3g_output_file(T_label, D_run)
        if output_file.exists():
            with open(output_file, 'rb') as f:
                results = pickle.load(f)
            print(f'Resuming {output_file.name}: {len(results)}/{FIG3G_N_BATCH_EVENTS} events already done')
        else:
            results = []

        for i in range(len(results), FIG3G_N_BATCH_EVENTS):
            t_event_start = time.time()
            results.append(run_single_cse_fig3g(
                event_index=i,
                D_run=D_run,
                seed=1000 + i,
                position_run=position_run,
                T_label=T_label,
                multiple_run=multiple_run,
            ))
            with open(output_file, 'wb') as f:
                pickle.dump(results, f)
            elapsed_min = (time.time() - t_event_start) / 60.0
            print(
                f'Saved partial result {i + 1}/{FIG3G_N_BATCH_EVENTS} '
                f'for T={T_label}, D={D_run} ({elapsed_min:.2f} min)'
            )

        print('Saved:', output_file)


## Figure 3(g) Grid Summary and Comparison


In [ ]:
# Summarize the full Figure 3(g) T-D grid and compare with the Qiwen control-z table.
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_PATH = globals().get('BASE_PATH', Path.cwd())
PathFigures = globals().get('PathFigures', EXPERIMENT_FIGURES_DIR)
PathFigures.mkdir(parents=True, exist_ok=True)

FIG3G_T_MULTIPLES = globals().get('FIG3G_T_MULTIPLES', {
    '[1.5T0]': 1.5,
    '[2T0]': 2.0,
    '[2.5T0]': 2.5,
    '[3T0]': 3.0,
    '[4T0]': 4.0,
})
FIG3G_D_VALUES = globals().get('FIG3G_D_VALUES', [0.02, 0.03, 0.04, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5])
FIG3G_N_BATCH_EVENTS = globals().get('FIG3G_N_BATCH_EVENTS', 40)
FIG3G_REGIME_REFERENCE = globals().get('FIG3G_REGIME_REFERENCE', 'nature')
FIG3G_CANDIDATE_POOL = globals().get('FIG3G_CANDIDATE_POOL', 'full')

QIWEN_FIG3G_CONTROL_Z_VALUES = globals().get('QIWEN_FIG3G_CONTROL_Z_VALUES', {
    '[1.5T0]': {0.02: 0.0, 0.03: 0.0, 0.04: 0.0, 0.05: 0.0, 0.1: 0.0, 0.2: 0.0, 0.3: 0.0, 0.4: 0.0, 0.5: 0.0},
    '[2T0]': {0.02: 0.0, 0.03: 0.0, 0.04: 0.0, 0.05: 0.0, 0.1: 0.0, 0.2: 0.0, 0.3: 0.0, 0.4: 0.0, 0.5: 0.0},
    '[2.5T0]': {0.02: 0.0, 0.03: 0.0, 0.04: 0.0, 0.05: 0.0, 0.1: 0.0, 0.2: 0.025, 0.3: 0.0, 0.4: 0.0, 0.5: 0.0},
    '[3T0]': {0.02: 0.0, 0.03: 0.0, 0.04: 0.0, 0.05: 0.0, 0.1: 0.45, 0.2: 0.45, 0.3: 0.275, 0.4: 0.35, 0.5: 0.275},
    '[4T0]': {0.02: 0.0, 0.03: 0.0, 0.04: 0.225, 0.05: 0.825, 0.1: 0.875, 0.2: 0.6, 0.3: 0.55, 0.4: 0.325, 0.5: 0.425},
})


def _fig3g_t_file_label(T_label):
    return T_label.strip('[]').replace('.', 'p').replace('T0', 'T0')


def _fig3g_output_file(T_label, D_run):
    T_file_label = _fig3g_t_file_label(T_label)
    output_suffix = (
        ''
        if (FIG3G_REGIME_REFERENCE == 'nature' and FIG3G_CANDIDATE_POOL == 'full')
        else f'_{FIG3G_REGIME_REFERENCE}_{FIG3G_CANDIDATE_POOL}'
    )
    return RESULTS_DIR / f'cse_qiwen_z_control_40_results_{T_file_label}_D{D_run}{output_suffix}{OUTPUT_TAG}.pkl'


def _legacy_4T0_output_file(D_run):
    return RESULTS_DIR / f'cse_qiwen_z_control_40_results_D{D_run}{OUTPUT_TAG}.pkl'


event_rows = []
aggregate_rows = []
for T_label, multiple_run in FIG3G_T_MULTIPLES.items():
    position_run = int(np.ceil(T_0 * multiple_run))
    for D_run in FIG3G_D_VALUES:
        result_file = _fig3g_output_file(T_label, D_run)
        if not result_file.exists() and T_label == '[4T0]':
            legacy_file = _legacy_4T0_output_file(D_run)
            if legacy_file.exists():
                result_file = legacy_file
        if not result_file.exists():
            print('Missing Figure 3(g) grid result file:', result_file)
            aggregate_rows.append({
                'T': T_label,
                'multiple_number': multiple_run,
                'position': position_run,
                'D': D_run,
                'n_events': 0,
                'success_ratio': np.nan,
                'mean_control_ratio': np.nan,
                'mean_false_alarm_ratio': np.nan,
                'mean_success_control_ratio': np.nan,
                'n_failures': np.nan,
                'result_file': str(result_file),
            })
            continue

        with open(result_file, 'rb') as f:
            rows = pickle.load(f)
        for row in rows:
            event_rows.append({
                'T': row.get('T_label', T_label),
                'multiple_number': row.get('multiple_number', multiple_run),
                'position': row.get('position', position_run),
                'D': D_run,
                'event': row['event_index'],
                'success': row['success'],
                'control_ratio': row['control_ratio'],
                'false_alarm_ratio': row['false_alarm_ratio'],
                'success_control_ratio': row['success_control_ratio'],
                'failure_cycle': row['failure_cycle'],
                'result_file': str(result_file),
            })
        aggregate_rows.append({
            'T': T_label,
            'multiple_number': multiple_run,
            'position': position_run,
            'D': D_run,
            'n_events': len(rows),
            'success_ratio': float(np.mean([row['success'] for row in rows])) if rows else np.nan,
            'mean_control_ratio': float(np.nanmean([row['control_ratio'] for row in rows])) if rows else np.nan,
            'mean_false_alarm_ratio': float(np.nanmean([row['false_alarm_ratio'] for row in rows])) if rows else np.nan,
            'mean_success_control_ratio': float(np.nanmean([row['success_control_ratio'] for row in rows])) if rows else np.nan,
            'n_failures': int(np.sum([not row['success'] for row in rows])) if rows else np.nan,
            'result_file': str(result_file),
        })

fig3g_event_table = pd.DataFrame(event_rows)
fig3g_aggregate_table = pd.DataFrame(aggregate_rows)
fig3g_success_matrix = fig3g_aggregate_table.pivot(index='T', columns='D', values='success_ratio')
fig3g_success_matrix = fig3g_success_matrix.reindex(index=list(FIG3G_T_MULTIPLES), columns=FIG3G_D_VALUES)

qiwen_fig3g_matrix = pd.DataFrame.from_dict(QIWEN_FIG3G_CONTROL_Z_VALUES, orient='index')
qiwen_fig3g_matrix = qiwen_fig3g_matrix.reindex(index=list(FIG3G_T_MULTIPLES), columns=FIG3G_D_VALUES)
fig3g_difference_matrix = fig3g_success_matrix - qiwen_fig3g_matrix

fig3g_event_table_file = RESULTS_DIR / 'cse_qiwen_z_control_40_fig3g_grid_events_nature_qiwenevents.csv'
fig3g_aggregate_table_file = RESULTS_DIR / 'cse_qiwen_z_control_40_fig3g_grid_aggregates_nature_qiwenevents.csv'
fig3g_success_matrix_file = RESULTS_DIR / 'cse_qiwen_z_control_40_fig3g_success_matrix_nature_qiwenevents.csv'
fig3g_difference_matrix_file = RESULTS_DIR / 'cse_qiwen_z_control_40_fig3g_minus_qiwen_matrix_nature_qiwenevents.csv'
fig3g_event_table.to_csv(fig3g_event_table_file, index=False)
fig3g_aggregate_table.to_csv(fig3g_aggregate_table_file, index=False)
fig3g_success_matrix.to_csv(fig3g_success_matrix_file)
fig3g_difference_matrix.to_csv(fig3g_difference_matrix_file)

try:
    display(fig3g_success_matrix)
    display(fig3g_difference_matrix)
except NameError:
    print('This-run success matrix:')
    print(fig3g_success_matrix)
    print('This-run minus Qiwen matrix:')
    print(fig3g_difference_matrix)

PAPER_TABLE_COLORS = {
    'ge_08': '#9c7a00',
    'ge_06': '#d1a000',
    'ge_04': '#ffe173',
    'ge_02': '#fff0b3',
    'gt_00': '#fff8dc',
    'zero': '#ffffff',
    'missing': '#f2f2f2',
}


def _paper_cell_color(value):
    if pd.isna(value):
        return PAPER_TABLE_COLORS['missing']
    value = float(value)
    if np.isclose(value, 0.0):
        return PAPER_TABLE_COLORS['zero']
    if value >= 0.8:
        return PAPER_TABLE_COLORS['ge_08']
    if value >= 0.6:
        return PAPER_TABLE_COLORS['ge_06']
    if value >= 0.4:
        return PAPER_TABLE_COLORS['ge_04']
    if value >= 0.2:
        return PAPER_TABLE_COLORS['ge_02']
    return PAPER_TABLE_COLORS['gt_00']


def _format_paper_value(value):
    if pd.isna(value):
        return ''
    value = float(value)
    if np.isclose(value, 0.0):
        return '0'
    return f'{value:.3f}'.rstrip('0').rstrip('.')


def _format_D_label(value):
    return f'{float(value):.2f}'.rstrip('0').rstrip('.')


def _draw_paper_table(ax, matrix, title):
    ax.axis('off')
    header = [r'T \\ D'] + [_format_D_label(D_run) for D_run in matrix.columns]
    table_text = [header]
    table_colors = [['#ffffff'] * len(header)]

    for T_label in matrix.index:
        row_text = [T_label]
        row_colors = ['#ffffff']
        for D_run in matrix.columns:
            value = matrix.loc[T_label, D_run]
            row_text.append(_format_paper_value(value))
            row_colors.append(_paper_cell_color(value))
        table_text.append(row_text)
        table_colors.append(row_colors)

    mpl_table = ax.table(
        cellText=table_text,
        cellColours=table_colors,
        cellLoc='center',
        loc='center',
        bbox=[0.0, 0.0, 1.0, 0.88],
    )
    mpl_table.auto_set_font_size(False)
    mpl_table.set_fontsize(8.4)
    mpl_table.scale(1.0, 1.16)

    for (row, col), table_cell in mpl_table.get_celld().items():
        table_cell.set_edgecolor('black')
        table_cell.set_linewidth(0.65)
        if row == 0 or col == 0:
            table_cell.set_text_props(weight='normal', color='black')
            table_cell.set_facecolor('#ffffff')
        elif table_cell.get_facecolor()[:3] == (0.611764705882353, 0.47843137254901963, 0.0):
            table_cell.set_text_props(color='white')

    ax.text(0.5, 0.945, title, ha='center', va='center', fontsize=10)
    return mpl_table


fig = plt.figure(figsize=(11.8, 4.5), constrained_layout=False)
grid = fig.add_gridspec(2, 2, height_ratios=[0.22, 1.0], hspace=0.12, wspace=0.08)
legend_ax = fig.add_subplot(grid[0, :])
legend_ax.axis('off')
legend_labels = [[r'$\geq 0.8$', r'$\geq 0.6$', r'$\geq 0.4$', r'$\geq 0.2$', r'$> 0$', r'$0$']]
legend_colors = [[
    PAPER_TABLE_COLORS['ge_08'],
    PAPER_TABLE_COLORS['ge_06'],
    PAPER_TABLE_COLORS['ge_04'],
    PAPER_TABLE_COLORS['ge_02'],
    PAPER_TABLE_COLORS['gt_00'],
    PAPER_TABLE_COLORS['zero'],
]]
legend_table = legend_ax.table(
    cellText=legend_labels,
    cellColours=legend_colors,
    cellLoc='center',
    loc='center',
    bbox=[0.38, 0.18, 0.24, 0.62],
)
legend_table.auto_set_font_size(False)
legend_table.set_fontsize(8.2)
for (row, col), table_cell in legend_table.get_celld().items():
    table_cell.set_edgecolor('black')
    table_cell.set_linewidth(0.65)
    if col == 0:
        table_cell.set_text_props(color='white')

axes = [fig.add_subplot(grid[1, 0]), fig.add_subplot(grid[1, 1])]
_draw_paper_table(axes[0], qiwen_fig3g_matrix, '(g) Qiwen control z')
_draw_paper_table(axes[1], fig3g_success_matrix, '(g) Diego control z')

fig3g_grid_png = PathFigures / 'cse_qiwen_z_control_fig3g_grid_comparison_nature_qiwenevents.png'
fig.savefig(fig3g_grid_png, dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)

print('Saved event table:', fig3g_event_table_file)
print('Saved aggregate table:', fig3g_aggregate_table_file)
print('Saved success matrix:', fig3g_success_matrix_file)
print('Saved difference matrix:', fig3g_difference_matrix_file)
print('Saved comparison figure:', fig3g_grid_png)
